# 7.11 · 鲁棒性与对抗样本 / Robustness & Adversarial Examples

> **课程定位 / Where this fits**
> 第 11 课，**Part 7 · 模型评估与优化**。
> Lesson 11, **Part 7 · Model Evaluation & Tuning**.
>
> 一个在测试集上 98% 准确率的模型，可能被一个**人眼几乎看不出的微小扰动**骗到完全失效——这就是**对抗样本**。它揭示了深度模型一个反直觉的脆弱性，在安全攸关场景（自动驾驶、人脸识别、内容审核）是真实威胁。这一课用最经典的 **FGSM 攻击**演示，并讲防御直觉。
> A model with 98% test accuracy can be fooled into total failure by a **perturbation almost invisible to humans** — an **adversarial example**. It reveals a counterintuitive fragility of deep models, a real threat in safety-critical settings (self-driving, face recognition, moderation). This lesson demonstrates the classic **FGSM attack** and defense intuition.
>
> 💼 **实战/面试视角**："对抗样本是什么 / 为什么模型这么脆弱 / 怎么防御" 偏安全/CV/深度学习岗。
> 💼 **Practical/interview angle:** "what are adversarial examples / why so fragile / how to defend" — security/CV/DL roles.

> 💡 **面试相关 / Interview-relevant**
> - "对抗样本是什么 / FGSM 原理"（出镜率 ★★★★）
> - "为什么深度模型对对抗扰动脆弱"（★★★★）
> - "对抗训练怎么防御"（★★★）
> - "鲁棒性 vs 准确率的权衡"（★★★）

---

## 学习目标 / Learning Objectives

1. 理解对抗样本：微小扰动让模型崩溃。
   Understand adversarial examples: tiny perturbations crash the model.
2. 实现 **FGSM 攻击**（沿损失梯度方向加扰动）。
   Implement the FGSM attack (perturb along the loss gradient).
3. 看模型准确率如何随扰动强度 ε 崩溃。
   See accuracy collapse as the perturbation strength ε grows.
4. 理解**对抗训练**这一防御及其代价。
   Understand adversarial training as a defense and its cost.

## 目录 / TOC
1. [先建直觉：为什么模型脆弱 ⭐](#1)
2. [🔢 数据 + 训练一个分类器](#2)
3. [FGSM 攻击 ⭐](#3)
4. [准确率随 ε 崩溃 ⭐](#4)
5. [防御：对抗训练 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 先建直觉：为什么模型脆弱 ⭐ / Intuition: Why Models Are Fragile

直觉上"图片改一点点，预测应该也只变一点点"。但事实是：在高维输入空间里，模型的决策边界离很多样本**其实很近**。攻击者只要**沿着'最能增大损失的方向'推一小步**，就能把样本推过边界、变成另一个类——而这一步在像素上小到人眼看不出。
Intuitively "change the image a little, the prediction should change a little". But in fact: in high-dimensional input space, the decision boundary is **actually very close** to many samples. An attacker only needs to **nudge along "the direction that most increases the loss"** to push a sample across the boundary into another class — a nudge so small in pixels it's invisible.

关键洞察（Goodfellow 2014）：脆弱性主要来自**模型在高维空间里的近似线性**。线性模型在每个方向上对扰动的累积响应 $\sum w_i \epsilon$ 在高维下能积累成很大的变化——所以**不是因为模型太复杂/非线性，恰恰因为它在局部太线性**。
The key insight (Goodfellow 2014): the fragility comes mainly from models being **approximately linear in high dimensions**. A linear model's accumulated response to a perturbation, $\sum w_i \epsilon$, can grow large across many dimensions — so it's **not because models are too complex/nonlinear, but precisely because they're too locally linear**.


<a id="2"></a>
## 2. 数据 + 训练一个分类器 / Data & a Classifier

用 **Digits**（8×8 手写数字）训一个小神经网络（用 PyTorch，因为 FGSM 需要对输入求梯度）。先确认它在干净测试集上很准。
We train a small neural net on **Digits** (8×8 handwritten digits) with PyTorch (FGSM needs gradients w.r.t. the input). First confirm it's accurate on clean test data.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
sns.set_theme(style="whitegrid")
torch.manual_seed(0); np.random.seed(0)

digits = load_digits()
X = digits.data / 16.0                                  # 像素归一化到 [0,1] / scale to [0,1]
y = digits.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
Xtr_t = torch.tensor(X_tr, dtype=torch.float32); ytr_t = torch.tensor(y_tr)
Xte_t = torch.tensor(X_te, dtype=torch.float32); yte_t = torch.tensor(y_te)

net = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10))   # 小 MLP
opt = torch.optim.Adam(net.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(120):
    opt.zero_grad(); loss = loss_fn(net(Xtr_t), ytr_t); loss.backward(); opt.step()

clean_acc = (net(Xte_t).argmax(1) == yte_t).float().mean().item()
print(f"干净测试集准确率 clean accuracy = {clean_acc:.3f}  (模型很准)")


<a id="3"></a>
## 3. FGSM 攻击 ⭐ / The FGSM Attack

**FGSM（快速梯度符号法，Fast Gradient Sign Method）** 是最经典的对抗攻击，一步到位：
**FGSM (Fast Gradient Sign Method)** is the classic one-shot attack:

$$\mathbf{x}_{adv} = \mathbf{x} + \varepsilon \cdot \text{sign}\big(\nabla_\mathbf{x} L(\mathbf{x}, y)\big)$$

直觉：$\nabla_\mathbf{x} L$ 是"损失对每个像素的敏感方向"，取它的**符号**（每个像素只 +ε 或 −ε），就是"用固定的小预算 ε，在每个像素上都往最能增大损失的方向推"。$\varepsilon$ 控制扰动强度——越大越容易骗到，但也越容易被人眼察觉。
Intuition: $\nabla_\mathbf{x} L$ is "the loss's sensitivity to each pixel"; taking its **sign** (each pixel ±ε) means "spend a fixed budget ε per pixel in the direction that most increases the loss". $\varepsilon$ controls strength — larger fools more easily but is more visible.

注意：梯度是对**输入**求的（不是对权重），所以模型权重不变，被改的是图片。
Note: the gradient is w.r.t. the **input** (not weights), so the model is unchanged; the image is what's modified.


In [ ]:
def fgsm(net, x, y, eps):
    x = x.clone().detach().requires_grad_(True)        # 要对输入 x 求梯度 → requires_grad
    loss = loss_fn(net(x), y)
    loss.backward()                                     # 反传得到 ∂loss/∂x
    x_adv = x + eps * x.grad.sign()                     # 沿梯度符号方向推一步(每像素 ±eps)
    return x_adv.clamp(0, 1).detach()                  # 裁回合法像素范围 [0,1]

eps = 0.15
X_adv = fgsm(net, Xte_t, yte_t, eps)
adv_acc = (net(X_adv).argmax(1) == yte_t).float().mean().item()
print(f"扰动强度 eps={eps}: 干净准确率 {clean_acc:.3f} → 对抗后 {adv_acc:.3f}  (崩溃!)")

# 可视化: 原图 / 扰动 / 对抗图, 人眼几乎看不出差别 / original vs perturbation vs adversarial
i = 0
orig = Xte_t[i].reshape(8,8).numpy(); adv = X_adv[i].reshape(8,8).numpy()
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(orig, cmap="gray_r"); axes[0].set_title(f"原图 → 预测 {net(Xte_t[i:i+1]).argmax(1).item()}"); axes[0].axis("off")
axes[1].imshow(adv-orig, cmap="RdBu"); axes[1].set_title(f"扰动 ×{1/eps:.0f}(放大看)"); axes[1].axis("off")
axes[2].imshow(adv, cmap="gray_r"); axes[2].set_title(f"对抗图 → 预测 {net(X_adv[i:i+1]).argmax(1).item()}"); axes[2].axis("off")
plt.suptitle("FGSM: 人眼看原图和对抗图几乎一样, 模型却预测错了"); plt.tight_layout(); plt.show()


<a id="4"></a>
## 4. 准确率随 ε 崩溃 ⭐ / Accuracy Collapses with ε

扫不同的扰动强度 ε，看模型准确率怎么塌。会看到一条**急剧下降**的曲线：很小的 ε 就能让 98% 的准确率掉到接近随机（10 类的随机基线是 10%）——这就是对抗脆弱性的可怕之处。
Sweep the perturbation strength ε and watch accuracy collapse. You'll see a **sharp drop**: a small ε takes 98% down toward random (the 10-class chance baseline is 10%) — the scary part of adversarial fragility.


In [ ]:
epsilons = [0.0, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3]
accs = []
for e in epsilons:
    Xa = fgsm(net, Xte_t, yte_t, e) if e > 0 else Xte_t
    accs.append((net(Xa).argmax(1) == yte_t).float().mean().item())

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epsilons, accs, "o-", lw=2)
ax.axhline(0.1, color="gray", ls="--", label="随机基线 chance (10类=10%)")
ax.set_xlabel("扰动强度 ε"); ax.set_ylabel("准确率 accuracy"); ax.legend()
ax.set_title("FGSM: 准确率随扰动强度急剧崩溃")
plt.tight_layout(); plt.show()
for e, a in zip(epsilons, accs): print(f"  ε={e:.2f}: 准确率 {a:.3f}")
print("→ 很小的扰动就让准确率从 ~0.97 跌到接近随机; 模型对对抗扰动极度脆弱")


<a id="5"></a>
## 5. 防御：对抗训练 + 小结 ⭐ / Defense: Adversarial Training

最有效的防御是**对抗训练(adversarial training)**：训练时不只喂干净样本，还**实时生成对抗样本一起训**——相当于让模型"见过攻击"，学会抵抗。下面对比对抗训练前后在对抗样本上的准确率。
The most effective defense is **adversarial training**: train not only on clean samples but also on **adversarial examples generated on the fly** — the model "has seen attacks" and learns to resist. Below we compare adversarial accuracy before/after.

**代价（鲁棒性-准确率权衡）**：对抗训练通常会**略降干净数据上的准确率**、且训练更慢。而且它只对训练时用的攻击类型鲁棒，**对更强的新攻击仍可能失效**——对抗攻防是一个持续的军备竞赛，至今没有完美防御。
**The cost (robustness-accuracy trade-off):** adversarial training usually **lowers clean accuracy slightly** and is slower. It's only robust to the attack type used in training and **may still fail against stronger new attacks** — adversarial attack/defense is an ongoing arms race with no perfect defense.


In [ ]:
# 对抗训练: 每步用当前模型生成对抗样本, 和干净样本一起训 / adversarial training
net2 = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10))
opt2 = torch.optim.Adam(net2.parameters(), lr=1e-2)
for epoch in range(120):
    opt2.zero_grad()
    Xadv_tr = fgsm(net2, Xtr_t, ytr_t, 0.15)                       # 用当前模型现造对抗样本
    loss = loss_fn(net2(Xtr_t), ytr_t) + loss_fn(net2(Xadv_tr), ytr_t)  # 干净 + 对抗 一起训
    loss.backward(); opt2.step()

eps = 0.15
clean2 = (net2(Xte_t).argmax(1) == yte_t).float().mean().item()
adv2 = (net2(fgsm(net2, Xte_t, yte_t, eps)).argmax(1) == yte_t).float().mean().item()
print(f"{'模型':<16} {'干净准确率':>10} {'对抗准确率(ε=0.15)':>18}")
print(f"{'普通训练':<16} {clean_acc:>10.3f} {adv_acc:>18.3f}")
print(f"{'对抗训练':<16} {clean2:>10.3f} {adv2:>18.3f}")
print("\n对抗训练大幅提升对抗准确率, 但干净准确率略降(鲁棒性-准确率权衡)")
print("注: 只对见过的攻击鲁棒; 对抗攻防是持续军备竞赛, 无完美防御")


```
对抗样本: 人眼几乎看不出的微小扰动, 让高准确率模型完全失效
脆弱根源: 模型在高维空间近似线性, 扰动沿梯度方向累积成大变化(不是太复杂, 是太线性)
FGSM: x_adv = x + ε·sign(∇_x L); 对输入求梯度, 取符号, 每像素推 ε
准确率随 ε 急剧崩溃: 很小扰动就跌到接近随机
防御: 对抗训练(训练时混入对抗样本); 代价=干净准确率略降+只防见过的攻击
鲁棒性-准确率权衡; 对抗攻防是持续军备竞赛, 无完美防御
```

### 💡 面试速查 / Interview cheat-sheet
1. **对抗样本 = 微小扰动骗崩模型**; 人眼看不出但模型分错。
   Adversarial examples = tiny perturbations that crash the model; invisible to humans.
2. **FGSM**: 沿损失对输入的梯度符号方向加 ε 扰动(一步攻击)。
   FGSM: perturb by ε along the sign of the input-gradient (one-shot).
3. **脆弱根源是高维近似线性**(不是太非线性)。
   Fragility comes from high-dim near-linearity (not excess nonlinearity).
4. **对抗训练**是主流防御; 代价=干净准确率略降。
   Adversarial training is the main defense; cost = slight clean-accuracy drop.
5. **鲁棒性-准确率权衡**; 无完美防御(军备竞赛)。
   Robustness-accuracy trade-off; no perfect defense (arms race).

### 下一节 / Next
**7.12 概念漂移**——模型上线后, 数据分布会随时间变化, 模型悄悄失效。怎么检测漂移(DDM/ADWIN)并触发重训。
**7.12 Concept Drift** — after deployment, data distributions shift over time and the model silently decays. How to detect drift (DDM/ADWIN) and trigger retraining.
